In [6]:
import numpy as np
import pysindy as ps

import sys, os
sys.path.insert(0, os.getcwd())
from ode_systems import SYSTEMS, list_systems


from ode_utils import (
    generate_data, simulate_from_coefficients,
    plot_phase_portrait, plot_trajectories, plot_coefficient_comparison,
    plot_noise_sweep, relative_l2_error, coefficient_error, precision_recall,
    print_metrics, metrics_dict, build_true_coef_matrix,
    print_discovered_equations,
)

In [7]:
# System selection — change this key to switch problems.
# Run list_systems() to see all available options.
SYSTEM_KEY = "lorenz"
SYSTEM     = SYSTEMS[SYSTEM_KEY]

# Initial condition (must have SYSTEM.n_dim entries)
X0 = [-8.0, 8.0, 27.0]

# Integration
T_SPAN      = (0.0, 20.0)   # (t_start, t_end)
DT          = 0.01           # timestep
NOISE_LEVEL = 0.00           # additive noise as fraction of signal RMS
SEED        = 42

# Candidate library
POLY_DEGREE       = 2        # polynomial degree
INCLUDE_SINE      = False    # add Fourier terms (useful for oscillators)
CUSTOM_LIBRARY_FN = None     # set to a callable () -> PySINDy library, or None

# Optimiser (STLSQ)
THRESHOLD = 0.05   # sparsity threshold
ALPHA     = 0.05   # ridge regularisation
MAX_ITER  = 20

# Differentiation
# Options: 'spline' | 'smoothed_finite_difference' | 'finite_difference'
DIFF_METHOD = "spline"
SPLINE_S    = 1e-3   # smoothing factor (spline only)

# Evaluation
T_EVAL_END         = 10.0   # trajectory comparison window
SWEEP_NOISE_LEVELS = [0.0, 0.01, 0.05, 0.1, 0.2]

RESULTS_FILE = f"results/sindy_{SYSTEM_KEY}_results.pkl"

In [30]:
t, X = generate_data(SYSTEM, X0, T_SPAN, DT,
                     noise_level=NOISE_LEVEL, seed=SEED)
print(X)

[[-8.00000000e+00  8.00000000e+00  2.70000000e+01]
 [-6.48640305e+00  7.80317052e+00  2.57255226e+01]
 [-5.13801768e+00  7.56258900e+00  2.46088107e+01]
 ...
 [ 9.55398203e-03  2.74242738e-01  1.56306754e+01]
 [ 3.47204549e-02  2.74321477e-01  1.52194264e+01]
 [ 5.76547823e-02  2.77589358e-01  1.48190649e+01]]


In [43]:
n = len(t)
x1 = X[0:int(n/2),:]
x2 = X[int(n/2):,:]
delta_12 = x1-x2
r12 = np.linalg.norm(delta_12,axis=1)[:,np.newaxis]
hat = delta_12 / r12
u = np.concatenate([r12,hat],axis=1)

# print(x1)
# print(x2)
# print(x1 - x2)
# print(x1.shape)
# print(x2.shape)
# print(delta_12.shape)
# print(np.linalg.norm(delta_12,axis=1))

print(u)



poly_lib = ps.PolynomialLibrary(degree=4)
vectorHat_lib = ps.IdentityLibrary()
lib = ps.GeneralizedLibrary(
    [poly_lib,vectorHat_lib],
    [[1,1]],
    [[3],[4,5,6]]
)


[[18.15739123 -0.8908823  -0.23033047  0.39150556]
 [16.72186015 -0.90119719 -0.29381371  0.31861753]
 [15.65759825 -0.90305327 -0.36192141  0.23131727]
 ...
 [12.85762786  0.54731661  0.79881148  0.24968931]
 [13.6558023   0.53978461  0.79194723  0.28539823]
 [14.50188511  0.53270671  0.78330561  0.32039956]]


In [44]:
optimizer = ps.STLSQ(threshold=THRESHOLD, alpha=ALPHA, max_iter=MAX_ITER)
diff = ps.differentiation.FiniteDifference()
model = ps.SINDy(
    feature_library=lib,
    optimizer=optimizer,
    differentiation_method=diff,
)
model.fit(
    X[0:int(n/2),:],
    t=DT,
    u=u
)
model.get_feature_names()

['1',
 'u0',
 'u0^2',
 'u0^3',
 'u0^4',
 'u1',
 'u2',
 'u3',
 '1 u1',
 '1 u2',
 '1 u3',
 'u0 u1',
 'u0 u2',
 'u0 u3',
 'u0^2 u1',
 'u0^2 u2',
 'u0^2 u3',
 'u0^3 u1',
 'u0^3 u2',
 'u0^3 u3',
 'u0^4 u1',
 'u0^4 u2',
 'u0^4 u3']

In [ ]:
from scipy.io import loadmat